# Context Model Training from Raw `.osz` (API)

This notebook trains only the context model using `train_api` (no CLI).
It keeps the same main settings as `train_context_raw_data.ipynb` and supports resume from `last.ckpt`.

In [1]:
from pathlib import Path

import torch

from src.model import (
    ArchitectureSpec,
    TrainingSpec,
    WandbConfig,
    build_training_artifacts,
    create_training_context,
    load_training_context_from_checkpoint,
    prepare_sample_data_artifacts,
    train_context,
)


In [2]:
# Same intent/defaults as the CLI notebook
raw_osz_dir = Path("sample_data_large/raw")
data_root = Path("sample_data_large")
training_dir = data_root / "training"
repo_root = Path.cwd()
checkpoints_dir = repo_root / "checkpoints" / "sample_large_context"

epochs = 10
batch_size = 32
num_workers = 4

# Aggressive speed-first context budget.
history_max_tokens = 128
retrieval_top_k = 1
retrieval_max_tokens_per_window = 12
retrieval_exclude_last_n_windows = 2
use_motif_retrieval = True
max_cached_charts = 2

# Runtime acceleration knobs.
precision = "auto"
pin_memory = True
persistent_workers = True
prefetch_factor = 4
architecture_name = "taiko_context_transformer"
run_name = "test_10epochs"
prepare_data = False
save_inference_every_n_steps = 1000
use_resume_if_available = True
use_wandb = False
wandb_log_every_batches = 100
wandb_notebook_name = "train_context_raw_data_api.ipynb"
wandb_api_key = ""
wandb_offline = False

if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

checkpoints_dir.mkdir(parents=True, exist_ok=True)
last_checkpoint = checkpoints_dir / "last.ckpt"

index_cache_dir = training_dir / "index_cache"
inference_snapshots_dir = checkpoints_dir / "inference_snapshots"

print(f"raw_osz_dir={raw_osz_dir}")
print(f"data_root={data_root}")
print(f"checkpoints_dir={checkpoints_dir}")
print(f"last_checkpoint={last_checkpoint}")
print(f"index_cache_dir={index_cache_dir}")
print(f"inference_snapshots_dir={inference_snapshots_dir}")
print(f"device={device}")


raw_osz_dir=sample_data_large\raw
data_root=sample_data_large
checkpoints_dir=c:\Users\28548\PythonNotebooks\taiko-diffusion\checkpoints\sample_large_context
last_checkpoint=c:\Users\28548\PythonNotebooks\taiko-diffusion\checkpoints\sample_large_context\last.ckpt
device=cuda


In [3]:
# Prepare data artifacts from raw .osz inputs.
if prepare_data:
    artifacts = prepare_sample_data_artifacts(
        osz_inputs=[str(raw_osz_dir)],
        data_root=data_root,
    )
else:
    artifacts = build_training_artifacts(data_root, checkpoints_dir=checkpoints_dir)
    print("Skipping raw-data preparation and reusing existing training artifacts.")
print(artifacts)

architecture_spec = ArchitectureSpec(
    name=architecture_name,
    history_max_tokens=history_max_tokens,
    retrieval_top_k=retrieval_top_k,
    retrieval_max_tokens_per_window=retrieval_max_tokens_per_window,
    retrieval_exclude_last_n_windows=retrieval_exclude_last_n_windows,
    use_motif_retrieval=use_motif_retrieval,
    max_cached_charts=max_cached_charts,
)
training_spec = TrainingSpec(
    epochs=epochs,
    batch_size=batch_size,
    num_workers=num_workers,
    device=device,
    precision=precision,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
    prefetch_factor=prefetch_factor,
)

wandb_config = None
if use_wandb:
    wandb_config = WandbConfig(
        enabled=True,
        run_name=run_name,
        log_every_n_batches=wandb_log_every_batches,
        notebook_name=wandb_notebook_name,
        offline=wandb_offline,
        api_key=wandb_api_key,
        mode_name_for_run=architecture_name,
    )


Unpacking .osz files (total): 100%|██████████| 100/100 [00:00<00:00, 12340.54file/s]

Unpack summary | total: 100 | unpacked: 0 | skipped existing: 100 | failed corrupt: 0
[INFO] Fast-skip chart before full parse: C:\Users\28548\PythonNotebooks\taiko-diffusion\sample_data_large\unpacked\2463969\Tose Proeski - Ako me poglednes vo oci (Ognjen3800) [aaeky's Deception].osu (non_taiko_mode_0)
[INFO] Fast-skip chart before full parse: C:\Users\28548\PythonNotebooks\taiko-diffusion\sample_data_large\unpacked\2463969\Tose Proeski - Ako me poglednes vo oci (Ognjen3800) [aaeky's Insane].osu (non_taiko_mode_0)
[INFO] Fast-skip chart before full parse: C:\Users\28548\PythonNotebooks\taiko-diffusion\sample_data_large\unpacked\2463969\Tose Proeski - Ako me poglednes vo oci (Ognjen3800) [sst3ky's Normal].osu (non_taiko_mode_0)
[INFO] Fast-skip chart before full parse: C:\Users\28548\PythonNotebooks\taiko-diffusion\sample_data_large\unpacked\2463969\Tose Proeski - Ako me poglednes vo oci (Ognjen3800) [Take's Hard].osu (non_taiko_mode_0)
[INFO] Fast-skip chart before full parse: C:\User

[INFO] Fast-skip chart before full parse: C:\Users\28548\PythonNotebooks\taiko-diffusion\sample_data_large\unpacked\2477885\Camellia - dfor the DELTA (Daycore) [dfor the DAYCORE].osu (non_taiko_mode_0)
[INFO] Fast-skip chart before full parse: C:\Users\28548\PythonNotebooks\taiko-diffusion\sample_data_large\unpacked\2477885\Camellia - dfor the DELTA (Daycore) [ffor the FAYEW].osu (non_taiko_mode_2)
[INFO] Fast-skip chart before full parse: C:\Users\28548\PythonNotebooks\taiko-diffusion\sample_data_large\unpacked\2477885\Camellia - dfor the DELTA (Daycore) [mfor the MONO].osu (non_taiko_mode_3)
[INFO] Fast-skip chart before full parse: C:\Users\28548\PythonNotebooks\taiko-diffusion\sample_data_large\unpacked\2478904\Eri Sasaki - Ring of Fortune (77hina) [Memories].osu (non_constant_bpm_[379.746835443, 405.4054054054])
[INFO] Fast-skip chart before full parse: C:\Users\28548\PythonNotebooks\taiko-diffusion\sample_data_large\unpacked\2479183\Erika - I Don't Know (Nightcore & Cut Ver.) (Mi

In [4]:
if use_resume_if_available and last_checkpoint.exists():
    context = load_training_context_from_checkpoint(
        last_checkpoint,
        data_root=data_root,
        device=device,
        checkpoints_dir=checkpoints_dir,
        index_cache_dir=index_cache_dir,
        history_max_tokens=history_max_tokens,
        retrieval_top_k=retrieval_top_k,
        retrieval_max_tokens_per_window=retrieval_max_tokens_per_window,
        retrieval_exclude_last_n_windows=retrieval_exclude_last_n_windows,
        use_motif_retrieval=use_motif_retrieval,
        max_cached_charts=max_cached_charts,
        precision=precision,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
        prefetch_factor=prefetch_factor,
    )
    print(f"Resuming from checkpoint: {last_checkpoint}")
else:
    context = create_training_context(
        data_root=data_root,
        architecture_spec=architecture_spec,
        training_spec=training_spec,
        checkpoints_dir=checkpoints_dir,
        index_cache_dir=index_cache_dir,
        history_max_tokens=history_max_tokens,
        retrieval_top_k=retrieval_top_k,
        retrieval_max_tokens_per_window=retrieval_max_tokens_per_window,
        retrieval_exclude_last_n_windows=retrieval_exclude_last_n_windows,
        use_motif_retrieval=use_motif_retrieval,
        max_cached_charts=max_cached_charts,
        precision=precision,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
        prefetch_factor=prefetch_factor,
    )
    print("Starting a fresh context-model training run.")

print(f"start_epoch={context.start_epoch}")
print(f"target_epochs={epochs}")
print(f"architecture={context.architecture_spec.name}")


[startup] index cache lookup...
[startup] index cache lookup done in 0.00s
[startup] manifest...
[startup] manifest done in 15.03s
[startup] splits...
[startup] splits done in 0.00s
[startup] indexes...
[startup] indexes done in 0.43s
[startup] index cache save...
[startup] index cache save done in 0.01s
[startup] vocab...
[startup] vocab done in 0.00s
[startup] dataset objects...
[startup] dataset objects done in 0.93s


c:\Users\28548\.conda\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting a fresh context-model training run.
start_epoch=1
target_epochs=10
architecture=taiko_context_transformer


In [5]:
context = train_context(
    context,
    epochs=epochs,
    log_every_n_batches=wandb_log_every_batches,
    wandb_config=wandb_config,
    save_inference_every_n_steps=save_inference_every_n_steps,
    inference_snapshots_dir=inference_snapshots_dir,
)

print("Training finished.")
print(f"last checkpoint: {(checkpoints_dir / 'last.ckpt').resolve()}")
print(f"best checkpoint: {(checkpoints_dir / 'best.ckpt').resolve()}")
print(f"inference snapshots dir: {inference_snapshots_dir.resolve()}")


[runtime] precision requested=auto resolved=bf16 autocast=1 scaler=0
[runtime] context_budget history_max_tokens=128 retrieval_top_k=1 retrieval_max_tokens_per_window=12 retrieval_exclude_last_n_windows=2 use_motif_retrieval=1


Training:   0%|          | 0/596 [00:00<?, ?it/s]c:\Users\28548\.conda\envs\pytorch\Lib\site-packages\torch\nn\functional.py:5476: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)


Epoch 1/10 | lr: 0.000100 | train loss: 1.0834 | val loss: 0.9723 | train samp/s: 239.88 | train tok/s: 41471.86


Epoch 2/10 | lr: 0.000100 | train loss: 0.8835 | val loss: 0.8935 | train samp/s: 262.29 | train tok/s: 45372.60


Epoch 3/10 | lr: 0.000100 | train loss: 0.8397 | val loss: 0.8610 | train samp/s: 274.96 | train tok/s: 47580.63


...

Epoch 6/10 | lr: 0.000100 | train loss: 0.7730 | val loss: 0.8198 | train samp/s: 280.73 | train tok/s: 48578.28


Epoch 7/10 | lr: 0.000100 | train loss: 0.7588 | val loss: 0.8072 | train samp/s: 277.37 | train tok/s: 47948.40


Epoch 8/10 | lr: 0.000100 | train loss: 0.7457 | val loss: 0.7925 | train samp/s: 278.93 | train tok/s: 48292.58


Epoch 9/10 | lr: 0.000100 | train loss: 0.7336 | val loss: 0.7760 | train samp/s: 281.50 | train tok/s: 48682.28


Epoch 10/10 | lr: 0.000100 | train loss: 0.7218 | val loss: 0.7757 | train samp/s: 283.01 | train tok/s: 48972.38
Training finished.
last checkpoint: C:\Users\28548\PythonNotebooks\taiko-diffusion\checkpoints\sample_large_context\last.ckpt
best checkpoint: C:\Users\28548\PythonNotebooks\taiko-diffusion\checkpoints\sample_large_context\best.ckpt
